# **1. Perkenalan Dataset**

## Dry Bean Dataset

**Sumber:** [Kaggle - Dry Bean Dataset](https://www.kaggle.com/datasets/muratkokludataset/dry-bean-dataset)  
**Referensi asli:** UCI ML Repository (Koklu & Ozkan, 2020)

Dataset ini berisi hasil pengukuran morfologi dari 13.611 biji kacang kering yang difoto menggunakan sistem visi komputer. Setiap biji diklasifikasikan ke dalam **7 kelas varietas kacang**:
- Seker, Barbunya, Bombay, Cali, Dermosan, Horoz, Sira

**Fitur (16 fitur numerik):**
| Fitur | Keterangan |
|---|---|
| Area | Luas area biji (pixel) |
| Perimeter | Keliling biji |
| MajorAxisLength | Panjang sumbu mayor |
| MinorAxisLength | Panjang sumbu minor |
| AspectRation | Rasio aspek |
| Eccentricity | Eksentrisitas |
| ConvexArea | Luas convex hull |
| EquivDiameter | Diameter ekuivalen |
| Extent | Rasio area terhadap bounding box |
| Solidity | Kepadatan |
| roundness | Kebulatan |
| Compactness | Kompaksi |
| ShapeFactor1-4 | Faktor bentuk 1-4 |

**Target:** `Class` — 7 kelas varietas kacang kering

**Task:** Multiclass Classification

# **2. Import Library**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

print('Library berhasil diimport!')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

# **3. Memuat Dataset**

In [ ]:
# Install openpyxl untuk baca file Excel (dataset asli .xlsx)
# !pip install openpyxl -q

# Load dataset
# Download dari: https://www.kaggle.com/datasets/muratkokludataset/dry-bean-dataset
# Letakkan file Dry_Bean_Dataset.xlsx di folder yang sama dengan notebook ini

df = pd.read_excel('../Dry_Bean_Dataset_raw/Dry_Bean_Dataset.xlsx')

print('=== INFO DATASET ===')
print(f'Shape: {df.shape}')
print(f'Jumlah baris: {df.shape[0]}')
print(f'Jumlah kolom: {df.shape[1]}')
print()
print('=== 5 BARIS PERTAMA ===')
df.head()

In [ ]:
# Cek tipe data dan info umum
print('=== INFO KOLOM ===')
df.info()
print()
print('=== STATISTIK DESKRIPTIF ===')
df.describe()

# **4. Exploratory Data Analysis (EDA)**

In [ ]:
# ===== 4.1 Distribusi Target =====
print('=== DISTRIBUSI KELAS TARGET ===')
class_dist = df['Class'].value_counts()
print(class_dist)
print()

plt.figure(figsize=(10, 5))
ax = class_dist.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribusi Kelas Dry Bean', fontsize=14)
plt.xlabel('Kelas')
plt.ylabel('Jumlah Sampel')
plt.xticks(rotation=45)
for p in ax.patches:
    ax.annotate(str(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 4.2 Missing Values =====
print('=== CEK MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# ===== 4.3 Duplikasi =====
print('=== CEK DUPLIKASI ===')
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')

In [ ]:
# ===== 4.4 Distribusi Fitur Numerik =====
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Fitur numerik ({len(numeric_cols)}): {numeric_cols}')

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Freq')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Distribusi Fitur Numerik', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 4.5 Deteksi Outlier dengan Boxplot =====
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col, fontsize=10)
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Boxplot Deteksi Outlier', fontsize=14)
plt.tight_layout()
plt.show()

# Hitung jumlah outlier per kolom (IQR method)
print('=== JUMLAH OUTLIER PER FITUR (IQR Method) ===')
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f'{col}: {outliers} outlier ({outliers/len(df)*100:.2f}%)')

In [ ]:
# ===== 4.6 Heatmap Korelasi =====
plt.figure(figsize=(14, 10))
corr_matrix = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5, annot_kws={'size': 7})
plt.title('Heatmap Korelasi Antar Fitur', fontsize=14)
plt.tight_layout()
plt.show()

# Cek fitur dengan korelasi tinggi (>0.95)
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > 0.95:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j],
                               corr_matrix.iloc[i, j]))
print(f'\nPasangan fitur dengan korelasi > 0.95:')
for c in high_corr:
    print(f'  {c[0]} & {c[1]}: {c[2]:.3f}')

In [ ]:
# ===== 4.7 Pairplot sample fitur utama =====
sample_features = ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'Class']
sns.pairplot(df[sample_features], hue='Class', plot_kws={'alpha': 0.4}, height=2.5)
plt.suptitle('Pairplot Fitur Utama per Kelas', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

In [ ]:
# ===== 5.1 Hapus Missing Values =====
print(f'Shape sebelum hapus missing: {df.shape}')
df_clean = df.dropna()
print(f'Shape setelah hapus missing: {df_clean.shape}')
print(f'Baris dihapus: {df.shape[0] - df_clean.shape[0]}')

In [ ]:
# ===== 5.2 Hapus Duplikasi =====
print(f'Shape sebelum hapus duplikat: {df_clean.shape}')
df_clean = df_clean.drop_duplicates()
print(f'Shape setelah hapus duplikat: {df_clean.shape}')

In [ ]:
# ===== 5.3 Penanganan Outlier (IQR Clipping) =====
# Gunakan clipping agar data tidak hilang terlalu banyak
df_no_outlier = df_clean.copy()

for col in numeric_cols:
    Q1 = df_no_outlier[col].quantile(0.25)
    Q3 = df_no_outlier[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_no_outlier[col] = df_no_outlier[col].clip(lower=lower, upper=upper)

print('Outlier berhasil ditangani dengan IQR Clipping')
print(f'Shape data: {df_no_outlier.shape}')

In [ ]:
# ===== 5.4 Label Encoding pada Target =====
le = LabelEncoder()
df_no_outlier['Class_encoded'] = le.fit_transform(df_no_outlier['Class'])

print('=== Mapping Label Encoding ===')
for i, cls in enumerate(le.classes_):
    print(f'  {cls} -> {i}')

In [ ]:
# ===== 5.5 Pisah Fitur dan Target =====
X = df_no_outlier[numeric_cols]
y = df_no_outlier['Class_encoded']

print(f'Shape X (fitur): {X.shape}')
print(f'Shape y (target): {y.shape}')
print(f'\nDistribusi kelas setelah preprocessing:')
print(y.value_counts())

In [ ]:
# ===== 5.6 Normalisasi Fitur (StandardScaler) =====
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=numeric_cols)

print('Normalisasi selesai. Statistik setelah scaling:')
print(X_scaled.describe().loc[['mean', 'std']].round(4))

In [ ]:
# ===== 5.7 Train-Test Split =====
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape : {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape : {y_test.shape}')
print(f'\nRasio train: {len(X_train)/len(X_scaled)*100:.1f}%')
print(f'Rasio test : {len(X_test)/len(X_scaled)*100:.1f}%')

In [ ]:
# ===== 5.8 Simpan Data yang Sudah Dipreprocessing =====
os.makedirs('Dry_Bean_Dataset_preprocessing', exist_ok=True)

# Gabung X_scaled dengan y untuk disimpan
df_preprocessed = X_scaled.copy()
df_preprocessed['Class'] = y.values

train_df = X_train.copy()
train_df['Class'] = y_train.values

test_df = X_test.copy()
test_df['Class'] = y_test.values

df_preprocessed.to_csv('Dry_Bean_Dataset_preprocessing/dry_bean_preprocessed.csv', index=False)
train_df.to_csv('Dry_Bean_Dataset_preprocessing/dry_bean_train.csv', index=False)
test_df.to_csv('Dry_Bean_Dataset_preprocessing/dry_bean_test.csv', index=False)

print('Dataset hasil preprocessing berhasil disimpan!')
print('File:')
print('  Dry_Bean_Dataset_preprocessing/dry_bean_preprocessed.csv')
print('  Dry_Bean_Dataset_preprocessing/dry_bean_train.csv')
print('  Dry_Bean_Dataset_preprocessing/dry_bean_test.csv')

## Ringkasan Preprocessing

| Tahap | Aksi | Keterangan |
|---|---|---|
| Missing Values | `dropna()` | Tidak ada missing, aman |
| Duplikasi | `drop_duplicates()` | Hapus baris identik |
| Outlier | IQR Clipping | Clip ke batas IQR |
| Label Encoding | `LabelEncoder` | Target Class → 0-6 |
| Normalisasi | `StandardScaler` | Mean=0, Std=1 |
| Split | `train_test_split` | 80% train, 20% test, stratified |